In [1]:
import cv2
import torch
import torchvision

In [2]:
from torch2trt import TRTModule

model_trt = TRTModule()
model_trt.load_state_dict(torch.load('road_following_model_trt.pth'))

collision_model_trt = TRTModule()
collision_model_trt.load_state_dict(torch.load('best_model_trt.pth'))

<All keys matched successfully>

In [3]:
from jetracer.nvidia_racecar import NvidiaRacecar

car = NvidiaRacecar()

In [4]:
import traitlets
from jetcam.csi_camera import CSICamera

camera = CSICamera(width=224, height=224, capture_fps=24)

In [ ]:
import torch
import torch.nn.functional as F
from utils import preprocess
import numpy
import threading
import time

STEERING_GAIN = 0.85
STEERING_BIAS = -0.05
BLOCKED_THRESHOLD = 0.1
THROTTLE_SPEED = 0.16
STOP_DURATION = 3.0
last_blocked_time = 0
stopped = False

prob_blocked = 0.0
shared_frame = None
lock = threading.Lock()

def frame_capture_thread_func():
    global shared_frame
    while True:
        frame = camera.read()
        with lock:
            shared_frame = frame
        time.sleep(0.01)

# Collision detection thread
def collision_thread_func():
    global prob_blocked
    while True:
        with lock:
            if shared_frame is None:
                continue
            image = shared_frame.copy()
        collision_input_image = preprocess(image).float()
        with torch.no_grad():
            output = collision_model_trt(collision_input_image)
            probs = F.softmax(output, dim=1)
            prob_blocked_local = float(probs[0, 0])
        with lock:
            prob_blocked = prob_blocked_local
        time.sleep(0.05)
        
# Start collision thread
threading.Thread(target=frame_capture_thread_func, daemon=True).start()
threading.Thread(target=collision_thread_func, daemon=True).start()

# Main loop
while True:
    with lock:
        if shared_frame is None:
            continue
        image = shared_frame.copy()

    input_image = preprocess(image).half()

    # Steering inference
    with torch.no_grad():
        output = model_trt(input_image).cpu().numpy().flatten()
        x = float(output[0])
    car.steering = x * STEERING_GAIN + STEERING_BIAS

    # Throttle logic
    current_time = time.time()

    with lock:
        is_blocked = prob_blocked >= BLOCKED_THRESHOLD

    if is_blocked and not stopped:
        stopped = True
        last_blocked_time = current_time
        print("Obstacle detected — stopping for 3 seconds")

    if stopped:
        if current_time - last_blocked_time < STOP_DURATION:
            car.throttle = 0
        else:
            stopped = False
            car.throttle = THROTTLE_SPEED
    else:
        car.throttle = THROTTLE_SPEED